# Practice 23 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — Basics

### Task 1: Real data — penguins

1. Load the dataset: `sns.load_dataset('penguins').dropna().reset_index(drop=True)`
2. Set a `(species, island)` MultiIndex. Sort it.
3. Retrieve all `'Chinstrap'` rows.
4. Get all `'Dream'` island rows across every species.
5. Mean `body_mass_g` per species — use NumPy.
6. Which `(species, island)` pair has the highest mean `bill_length_mm`?

In [10]:
# Your code here
penguins = sns.load_dataset('penguins').dropna().reset_index(drop = True)
penguins = penguins.set_index(['species', 'island']).sort_index()
c = penguins.loc['Chinstrap']
idx = pd.IndexSlice
d = penguins.loc[idx[:,'Dream'],:]
m = penguins.groupby(level = 'species')['body_mass_g'].mean()
m.index[np.argmax(m)]
b = penguins.groupby(level = ['species','island'])['bill_length_mm'].mean()
b.index[np.argmax(b)]


('Chinstrap', 'Dream')

---
## Level 2 — Transformations

### Task 2: Survey responses — stack / unstack / .xs()

8 respondents answered 4 questions (scores 1–5). The DataFrame is in wide format with `respondent` as the index.

1. Stack to long format. Name the value column `'score'`. Store as `survey_long`.
2. Add `log_score` using NumPy.
3. Add `above_mid`: `True` if `score > 3`, using `np.where`.
4. Extract all `'Q3'` responses across every respondent.
5. Unstack back to wide format (scores only).
6. Which respondent has the highest total score across all questions?

In [18]:
# Starter data — don't change this
np.random.seed(3)
survey = pd.DataFrame({
    'respondent': [f'R{i:02d}' for i in range(1, 9)],
    'group':      np.random.choice(['Control', 'Treatment'], size=8),
    'Q1': np.random.randint(1, 6, size=8),
    'Q2': np.random.randint(1, 6, size=8),
    'Q3': np.random.randint(1, 6, size=8),
    'Q4': np.random.randint(1, 6, size=8),
}).set_index('respondent').drop(columns='group')

# Your code here
survey_long = survey.stack().to_frame('score')
survey_long['log_score'] = np.log(survey_long['score'])
survey_long['above_mid'] = np.where(survey_long['score']>3, True, False)
survey_long.xs('Q3', level=1)
w = survey_long['score'].unstack()
survey_long.groupby(level='respondent')['score'].sum().idxmax()




'R08'

### Task 3: Fitness tracker — .pipe()

Write all three functions yourself, then chain them with `.pipe()`:

- `add_pace(df)`: `pace` = `duration_min / distance_km`
- `add_intensity(df)`: `intensity` = `calories / duration_min`
- `flag_goal_met(df)`: `goal_met` = `True` if `calories > calorie_goal`

Then:
- What fraction of workouts met the goal? Use NumPy.
- Which `workout_type` has the highest mean `intensity`?

In [22]:
# Starter data — don't change this
np.random.seed(11)
workouts = pd.DataFrame({
    'workout_id':   [f'W{i:02d}' for i in range(1, 9)],
    'workout_type': np.random.choice(['Running', 'Cycling', 'HIIT', 'Yoga'], size=8),
    'duration_min': np.random.randint(20, 90, size=8),
    'distance_km':  np.round(np.random.uniform(2, 15, size=8), 1),
    'calories':     np.random.randint(150, 700, size=8),
    'calorie_goal': np.random.randint(300, 600, size=8),
})

# Your code here
def add_pace(df):
    df['pace'] = df['duration_min']/df['distance_km']
    return df
def add_intensity(df):
    df['intensity'] = df['calories']/df['duration_min']
    return df
def flag_goal_met(df):
    df['goal_met'] = df['calories']>df['calorie_goal']
    return df

res = (
    workouts
    .pipe(add_pace)
    .pipe(add_intensity)
    .pipe(flag_goal_met)
)
np.sum(res['goal_met'])/res.shape[0]
res.groupby('workout_type')['intensity'].mean().idxmax()


'Cycling'

### Task 4: .apply() + .str + .rank()

1. Add `title_lower`: book titles in lowercase.
2. Filter to books where `title` contains `'the'` (case-insensitive).
3. Extract the numeric part of `book_id` (e.g. `'B04'` → `'04'`). Store as `num`.
4. Add `price_rank`: rank by `price`, most expensive first.
5. Add `genre_rating_rank`: rank within each `genre` by `rating`, highest first.
6. Add `tier` (row-wise): `'Bestseller'` if `rating >= 4.5` and `copies_sold >= 5000`, `'Popular'` if either, else `'Standard'`.
7. Show only the top-ranked book per genre.

In [24]:
# Starter data — don't change this
np.random.seed(29)
books = pd.DataFrame({
    'book_id':    [f'B{i:02d}' for i in range(1, 9)],
    'title':      ['The Midnight Sky', 'Brave New Worlds', 'Into the Silence',
                   'Signal and Noise', 'The Last Garden', 'Atomic Habits',
                   'The Ocean Deep', 'Quiet Algorithms'],
    'genre':      ['Fiction', 'Sci-Fi', 'Fiction', 'Non-Fiction', 'Fiction', 'Non-Fiction', 'Fiction', 'Sci-Fi'],
    'price':      np.round(np.random.uniform(8, 35, size=8), 2),
    'rating':     np.round(np.random.uniform(3.0, 5.0, size=8), 2),
    'copies_sold': np.random.randint(500, 9000, size=8),
})

# Your code here
books['title_lower'] = books['title'].str.lower()
books[books['title'].str.contains('the',case=False)]
books['num'] = books['book_id'].str[1:]
books['price_rank'] = books['price'].rank(ascending=False)
books['genre_rating_rank'] =books.groupby('genre')['rating'].rank(ascending=False)
books['tier'] = books.apply(lambda row: 'Bestseller' if row['rating']>=4.5 and row['copies_sold']>=5000
                            else 'Popular' if row['rating']>=4.5 or row['copies_sold']>=5000
                            else 'Standard', axis = 1)
books[books['genre_rating_rank']==1]



,book_id,title,genre,price,rating,copies_sold,title_lower,num,price_rank,genre_rating_rank,tier
0,B01,The Midnight Sky,Fiction,31.32,4.54,3476,the midnight sky,01,1.0,1.0,Popular
1,B02,Brave New Worlds,Sci-Fi,15.69,4.47,1535,brave new worlds,02,7.0,1.0,Standard
3,B04,Signal and Noise,Non-Fiction,28.61,4.44,1527,signal and noise,04,3.0,1.0,Standard


---
## Level 3 — Aggregation

### Task 5: Merge + transform + rank

1. Inner join `applicants` with `postings` on `role_id`.
2. Add `dept_avg_salary`: mean `offered_salary` per `department` using `transform`.
3. Add `above_avg`: `True` if `offered_salary > dept_avg_salary`.
4. Add `overall_rank`: rank by `offered_salary`, highest first.
5. Add `dept_rank`: rank within each `department` by `offered_salary`, highest first.
6. Add `fit` (row-wise): `'Strong'` if `years_exp >= 7` and `skills_match >= 4`, `'Possible'` if either, else `'Weak'`.
7. Which `department` has the most `'Strong'` applicants?

In [34]:
# Starter data — don't change this
np.random.seed(44)
applicants = pd.DataFrame({
    'applicant_id':   [f'A{i:02d}' for i in range(1, 11)],
    'role_id':        np.random.choice([f'R{i:02d}' for i in range(1, 5)], size=10),
    'years_exp':      np.random.randint(1, 12, size=10),
    'skills_match':   np.random.randint(1, 6, size=10),
    'offered_salary': np.random.randint(50000, 150000, size=10),
})

postings = pd.DataFrame({
    'role_id':    [f'R{i:02d}' for i in range(1, 5)],
    'department': ['Engineering', 'Product', 'Design', 'Marketing'],
    'level':      ['Senior', 'Mid', 'Junior', 'Senior'],
})

# Your code here

ij = pd.merge(
    applicants,
    postings,
    on='role_id',
    how='inner'
)
ij['dept_avg_salary'] = ij.groupby('department')['offered_salary'].transform('mean')
ij['above_avg'] = ij['offered_salary']>ij['dept_avg_salary']
ij['overall_rank'] = ij['offered_salary'].rank( ascending=False)
ij['dept_rank'] = ij.groupby('department')['offered_salary'].rank(ascending=False)
ij['fit'] = ij.apply(lambda row: 'Strong' if row['years_exp']>=7 and row['skills_match']>=4 
                     else 'Possible' if row['years_exp']>7 or row['skills_match']>=4
                     else 'Weak', axis = 1)
ij.groupby('department')['fit'].apply(lambda x: (x=='Strong').sum()).idxmax()



'Marketing'

---
## Level 4 — Real-world

### Task 6: Tips — open analysis

Load the tips dataset: `sns.load_dataset('tips')`

Answer these questions using whatever methods you've learned:

1. Which `(day, time)` combination has the highest mean tip percentage? (tip as a % of total_bill)
2. Build a `.pipe()` pipeline — write at least two functions yourself and chain them.
3. Pivot mean tip percentage by `day` (rows) and `time` (columns). Stack the result, then use `.xs()` to extract just `'Dinner'`.
4. Do smokers or non-smokers tip more variably? Use NumPy.

In [46]:
# Your code here
tips = sns.load_dataset('tips')
tips['pct'] = tips['tip']/tips['total_bill']
tips.groupby(['day','time'])['pct'].mean().idxmax()

def flag_big_meal(df):
    df['big_meal'] = df['total_bill']>50
    return df
def flag_big_tip(df):
    df['big_tip'] = df['pct']>0.2
    return df


res = tips.pipe(flag_big_meal).pipe(flag_big_tip)
p = tips.pivot_table(
    index = 'day',
    columns= 'time',
    values='pct'
).stack().to_frame('pct').xs('Dinner', level = 'time')


v = tips.groupby('smoker', observed=True)['tip'].agg(np.std)
v.index[np.argmax(v)]

/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_12842/1800638306.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby(['day','time'])['pct'].mean().idxmax()
/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_12842/1800638306.py:15: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  p = tips.pivot_table(
/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_12842/1800638306.py:22: FutureWarning: The provided callable <function std at 0x7f9499c6a550> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std"

'Yes'